In [201]:
import numpy as np
import pandas as pd


In [202]:
data = {
    'feature1': [np.random.randint(0, 10) for _ in range(20)],
    'feature2': [np.random.randn() for _ in range(20)],
    'label': [np.random.randint(0, 2) for _ in range(20)]
}

df = pd.DataFrame(data)
print(df)

    feature1  feature2  label
0          1  0.508994      1
1          4 -0.683074      0
2          3 -0.394072      1
3          1 -0.250737      1
4          1 -0.647292      1
5          6  0.848767      1
6          5 -0.904932      1
7          7  0.566961      1
8          3 -1.201637      1
9          5  1.231281      0
10         4  1.217296      1
11         8 -0.260015      0
12         6  0.361555      1
13         8 -2.282147      1
14         0 -0.461322      0
15         5 -0.666481      0
16         0 -0.624301      0
17         7  0.063250      0
18         6  0.158032      0
19         1 -0.457998      1


In [203]:
def Ent(label):
    prob = np.bincount(label) / len(label)
    res = np.array([p * np.log2(p) if p != 0 else 0 for p in prob])
    return -np.sum(res)

def Gain(label, attr, attr_val):
    gain = Ent(label)
    for val in attr_val:
        label_temp = label[data[:, attr] == val]
        if len(label_temp) == 0:
            continue
        gain -= len(label_temp) / len(label) * Ent(label_temp)
    return gain

def discrete(df, attrs):
    label = df['label'].to_numpy()
    for attr in attrs:
        arr = df[attr].to_numpy()
        ix = np.argsort(arr)
        arr = arr[ix]
        label_temp = label[ix]

        mode = np.array([arr[0]] + [(arr[i] + arr[i + 1])/2 for i in range(len(arr) - 1)] + [arr[-1]])
        print(mode)
        print(label_temp)

        gain0 = Ent(label_temp)
        gains = []
        for m in mode:
            label_le = label_temp[arr <= m]
            label_gt = label_temp[arr > m]

            if len(label_le) == 0 or len(label_gt) == 0:
                gains.append(0)

            gain = gain0 - len(label_le) / len(label) * Ent(label_le) - len(label_gt) / len(label) * Ent(label_gt)
            gains.append(gain)

        ix = np.argmax(gains)
        opt_split = mode[ix]

        df[attr] = df[attr].apply(lambda x: f'<={opt_split:5.2}' if x <= opt_split else f'>{opt_split:5.2}')
        # df[attr] = df[attr].apply(lambda x: f'le{opt_split:5.2}' if x <= opt_split else f'gt{opt_split:5.2}')

    return df

df = discrete(df, ['feature2'])
print(df)

[-2.28214686 -1.74189211 -1.05328447 -0.79400303 -0.67477771 -0.6568863
 -0.63579609 -0.54281143 -0.45966034 -0.42603534 -0.32704362 -0.2553759
 -0.09374341  0.11064113  0.25979377  0.43527459  0.53797739  0.70786391
  1.03303144  1.22428823  1.23128052]
[1 1 1 0 0 1 0 0 1 1 0 1 0 0 1 1 1 1 1 0]
    feature1 feature2  label
0          1   >-0.79      1
1          4   >-0.79      0
2          3   >-0.79      1
3          1   >-0.79      1
4          1   >-0.79      1
5          6   >-0.79      1
6          5  <=-0.79      1
7          7   >-0.79      1
8          3  <=-0.79      1
9          5   >-0.79      0
10         4   >-0.79      1
11         8   >-0.79      0
12         6   >-0.79      1
13         8  <=-0.79      1
14         0   >-0.79      0
15         5   >-0.79      0
16         0   >-0.79      0
17         7   >-0.79      0
18         6   >-0.79      0
19         1   >-0.79      1
